In [ ]:
import pandas as pd

# 假设数据：每年增加一台（2035-01-01, 2036-01-01, 2037-01-01）
assumptions_data = [
    {
        "unit_id": "unit_1",
        "nameplate_mwe": 470,
        "net_delivery_factor": 0.90,
        "commissioning_date": "2035-01-01",
        "planned_outage_window": 18,
        "forced_outage_rate": 0.05,
        "smr_case": "staggered_commissioning"
    },
    {
        "unit_id": "unit_2",
        "nameplate_mwe": 470,
        "net_delivery_factor": 0.90,
        "commissioning_date": "2036-01-01",
        "planned_outage_window": 18,
        "forced_outage_rate": 0.05,
        "smr_case": "staggered_commissioning"
    },
    {
        "unit_id": "unit_3",
        "nameplate_mwe": 470,
        "net_delivery_factor": 0.90,
        "commissioning_date": "2037-01-01",
        "planned_outage_window": 18,
        "forced_outage_rate": 0.05,
        "smr_case": "staggered_commissioning"
    }
]

# 转换为 DataFrame 并保存
df_assumptions = pd.DataFrame(assumptions_data)
df_assumptions.to_csv("smr_assumptions.csv", index=False)
print("Step 1: smr_assumptions.csv 已更新。")

In [ ]:
import pandas as pd

# 1. 设置严格的时间范围（从 2010 年元旦到 2045 年除夕）
start_date = "2010-01-01 00:00:00"
end_date = "2045-12-31 23:00:00"

# 2. 生成每小时的时间序列 (Frequency='H' 代表 Hourly)
# pandas 会自动处理其中的闰年（如 2012, 2016, 2032 等）
timestamps = pd.date_range(start=start_date, end=end_date, freq="H")

# 3. 创建 DataFrame
df_calendar = pd.DataFrame({"timestamp_utc": timestamps})

# 4. 提取时间特征 (按照 Owner 1 的任务要求，这些是后续逻辑判断的关键)
df_calendar["year"] = df_calendar["timestamp_utc"].dt.year
df_calendar["month"] = df_calendar["timestamp_utc"].dt.month
df_calendar["day"] = df_calendar["timestamp_utc"].dt.day
df_calendar["hour"] = df_calendar["timestamp_utc"].dt.hour

# 5. 打印信息进行验证
print(f"Calendar successfully generated!")
print(f"Total rows: {len(df_calendar)}") # 应该是 315,576 行左右
print(df_calendar.head())

# 6. 保存为 Parquet 格式 (使用 pyarrow 或 fastparquet 引擎)
# 确保文件名与任务说明中要求的一字不差
df_calendar.to_parquet("calendar_hourly_2010_2045.parquet", index=False)

print("\nFile saved as: calendar_hourly_2010_2045.parquet")

In [ ]:
# smr小时级发电
import pandas as pd
import numpy as np

# ==========================================
# 1. 加载输入数据
# ==========================================
# 加载你之前生成的日历“画布”
df_full_calendar = pd.read_parquet("calendar_hourly_2010_2045.parquet")

# 按照任务说明要求，过滤出 2030-2045 的目标区间
df_base = df_full_calendar[df_full_calendar["year"] >= 2030].copy()

# 加载你生成的 SMR 假设表
assumptions = pd.read_csv("smr_assumptions.csv")

library_list = []

# ==========================================
# 2. 逐个机组应用发电逻辑
# ==========================================
for _, row in assumptions.iterrows():
    unit_df = df_base.copy()
    unit_df["unit_id"] = row["unit_id"]
    unit_df["smr_case"] = row["smr_case"]
    unit_df["nameplate_mw"] = row["nameplate_mwe"]

    # A. 投产逻辑 (Commissioning Logic)
    # 遵循 "No output ramp" 决策：投产日前为 False，投产日瞬间变为 True
    unit_df["is_commissioned"] = unit_df["timestamp_utc"] >= pd.to_datetime(row["commissioning_date"])

    # B. 计划停机 (Planned Outage) 逻辑
    # 模拟 18 天（432小时）的维护周期
    # 我们通过设置 offset 错开三台机组的维护时间，避免电力系统同时停机[cite: 1]
    offset_map = {"unit_1": 5000, "unit_2": 10000, "unit_3": 15000}
    cycle_hours = 18 * 30 * 24  # 假设每18个月为一个大周期[cite: 2]
    maintenance_hours = row["planned_outage_window"] * 24 # 18天对应的小时数[cite: 1]

    # 计算当前小时在周期中的位置
    relative_hours = (np.arange(len(unit_df)) + offset_map[row["unit_id"]]) % cycle_hours
    unit_df["planned_outage"] = relative_hours < maintenance_hours

    # C. 强制故障 (Forced Outage) 逻辑
    # 使用 5% 的故障率进行随机模拟[cite: 2]
    np.random.seed(42) # 固定随机种子确保结果可复现[cite: 2]
    unit_df["forced_outage"] = np.random.random(len(unit_df)) < row["forced_outage_rate"]

    # D. 可用性判断 (is_available)
    # 必须同时满足：已投产、不在计划维护中、且没有突发故障[cite: 1]
    unit_df["is_available"] = (
        unit_df["is_commissioned"] &
        (~unit_df["planned_outage"]) &
        (~unit_df["forced_outage"])
    )

    # E. 计算最终交付电量 (delivered_mw)
    # 逻辑：如果可用，则输出 = 额定容量 * 净交付因子；否则为 0[cite: 1, 2]
    unit_df["delivered_mw"] = np.where(
        unit_df["is_available"],
        row["nameplate_mwe"] * row["net_delivery_factor"],
        0.0
    )

    library_list.append(unit_df)

# ==========================================
# 3. 合并数据并保存交付物
# ==========================================
# 合并为长表格式 (Long-format library)[cite: 1]
final_library = pd.concat(library_list, ignore_index=True)

# 确保所有电力列使用 _mw 后缀并符合 float64 精度要求[cite: 1]
final_library["delivered_mw"] = final_library["delivered_mw"].astype(float)

# 保存为 Parquet 格式，供 Owner 2 使用[cite: 1]
final_library.to_parquet("smr_hourly_library_2030_2045.parquet", index=False)

print("Owner 1 任务完成！")
print(f"生成的库文件包含 {len(final_library)} 行数据。")
print(final_library.head())

In [ ]:
# part2:

In [ ]:
import pandas as pd

# ==========================================
# 1. 读取 Owner 1 的输出数据
# ==========================================
df_long = pd.read_parquet("smr_hourly_library_2030_2045.parquet")

# ==========================================
# 2. 长表转宽表 (Pivoting)
# ==========================================
# 将 unit_id 展开为列，每一行是一个时间戳
df_wide = df_long.pivot_table(
    index=["timestamp_utc", "smr_case", "year"],
    columns="unit_id",
    values="delivered_mw",
    fill_value=0.0
).reset_index()

# 规范化列名
df_wide = df_wide.rename(columns={
    "unit_1": "unit_1_mw",
    "unit_2": "unit_2_mw",
    "unit_3": "unit_3_mw"
})

# 确保三台机组列都存在，即使某台机组在初期年份还未投产
for unit in ["unit_1_mw", "unit_2_mw", "unit_3_mw"]:
    if unit not in df_wide.columns:
        df_wide[unit] = 0.0

# 计算整个机队的总 MW (Total Fleet)
df_wide["total_fleet_mw"] = (
    df_wide["unit_1_mw"] +
    df_wide["unit_2_mw"] +
    df_wide["unit_3_mw"]
)

# ==========================================
# 3. 情景克隆与标记 (Scenario Duplication)
# ==========================================
# 复制两份：Holistic Transition 和 Electric Engagement
df_ht = df_wide.copy()
df_ht["fes_scenario"] = "Holistic Transition"

df_ee = df_wide.copy()
df_ee["fes_scenario"] = "Electric Engagement"

# 合并生成最终的机队场景数据集
df_fleet = pd.concat([df_ht, df_ee], ignore_index=True)

# ==========================================
# 4. 调整列的顺序并保存
# ==========================================
cols_order = [
    "timestamp_utc", "fes_scenario", "smr_case", "year",
    "unit_1_mw", "unit_2_mw", "unit_3_mw", "total_fleet_mw"
]
df_fleet = df_fleet[cols_order]

# 保存为 Parquet 格式，供下游或下一步使用
df_fleet.to_parquet("smr_hourly_fleet_scenarios.parquet", index=False)

print("Owner 2 任务完成！生成的机队场景数据头部如下：")
print(df_fleet.head())

In [ ]:
import pandas as pd

df = pd.read_parquet("smr_hourly_fleet_scenarios.parquet")

print(f"总行数: {len(df)}")
# 应该包含这两种情景下的数据
print(df["fes_scenario"].value_counts())

# 检查 2035 年 unit_1 投产后某一时刻的发电量
sample = df[
    (df["timestamp_utc"] == "2035-01-01 00:00:00") &
    (df["fes_scenario"] == "Holistic Transition")
]
print("\n样本检查：")
print(sample[["timestamp_utc", "fes_scenario", "unit_1_mw", "total_fleet_mw"]])